## Visualize ensemble reconstruction results

In [ ]:
import jax
jax.config.update('jax_platform_name', 'cpu')
import os 
os.environ['CUDA_VISIBLE_DEVICES'] = ''

import sys 
sys.path.insert(1, '../')

import plotly.graph_objs as go
from plotly.offline import iplot
import plotly.io as pio

from jax import numpy as jnp
import numpy as np

from scipy.ndimage import gaussian_filter

import matplotlib.pyplot as plt

from tqdm.notebook import tqdm

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.offline import init_notebook_mode

init_notebook_mode(connected=True) 

In [ ]:
pm_seed = 2
num_its = 1

# i_list = range(64)
i_list = range(128)
# i_list = range(1)
# i_list = list(range(0, 0+num_its)) + list(range(25, 25+num_its)) + list(range(50, 50+num_its)) + list(range(75, 75+num_its))
save_dirs = ['../../exp/vol_jwst%d_L_2_Lz_5_proj_gg_shape_sigma_0.25_lam_0.001/%d'%(pm_seed, i) for i in i_list]

# wiener_recon = jnp.load('../../wiener_recon/pm%d_wiener.npy'%pm_seed).transpose((1,2,0))
wiener_recon = jnp.load(f'/srv/tmp/brandon/abacus/wiener_recon/wiener_jwst_{pm_seed}.npy').transpose(1, 2, 0)
print(wiener_recon.shape)
ddpm_recon = jnp.load(f'/srv/tmp/brandon/abacus/wiener_recon/ddpm_jwst_{pm_seed}.npy')[:, 30:99, :69] .transpose(1, 2, 0)
ddpm_samples = jnp.load(f'/srv/tmp/brandon/abacus/wiener_recon/ddpm_jwst_{pm_seed}_samples.npy')[:, :, 30:99, :69].transpose(0, 2, 3, 1)
density_paths = [save_dir + '/density/density_500.p' for save_dir in save_dirs]
# density_paths_2 = [save_dir + '/density/density_500.p' for save_dir in save_dirs_2]

res = 69

In [ ]:
def mean_subtract(vol, axis=(0,1)): 
    return vol - jnp.mean(vol, axis=axis, keepdims=True)

densities = [mean_subtract(jnp.load(density_path).reshape(res, res, 20)) for density_path in density_paths]
densities_arr = jnp.array(densities)
densities_mean = jnp.mean(densities_arr, axis=0)

density_gt = jnp.load(save_dirs[-1] + '/../density_gt.p').reshape(res, res, 20)

pws = jnp.load(save_dirs[0] + '/../pws.npy')
pws_ein = pws.reshape(-1, res, res, 20)

density_gt = mean_subtract(density_gt)
ddpm_recon = mean_subtract(ddpm_recon)
wiener_recon = mean_subtract(wiener_recon)

print(densities_mean.shape)
print(ddpm_recon.shape)
print(wiener_recon.shape)

In [ ]:
X, Y, Z = np.mgrid[:69, :69, :12]
X = 69 - X
Z = 12 - Z

def zoom_out_same_pov(fig, scale=1.3):
    cam_json = (fig.layout.scene.camera.to_plotly_json()
                if fig.layout.scene.camera is not None else {})
    eye = cam_json.get("eye", {"x": 1.25, "y": 1.25, "z": 1.25})
    new_eye = {k: float(eye.get(k, 1.25)) * scale for k in ("x","y","z")}
    fig.update_layout(scene_camera=dict(eye=new_eye))

In [ ]:
density_gt_nofilter = gaussian_filter(density_gt, (2, 2, 0.), mode='constant')[..., :12]
density_gt_filter = gaussian_filter(density_gt, (2, 2, 4.), mode='constant')[..., :12]
wiener_recon_filter = gaussian_filter(wiener_recon, (2, 2, 0))[..., :12]
ddpm_recon_filter = gaussian_filter(ddpm_recon, (2, 2, 0))[..., :12]
neur_recon_filter = gaussian_filter(densities_mean, (2, 2, 0))[..., :12]

ddpm_sample_filter = gaussian_filter(ddpm_samples[0], (2, 2, 0))[..., :12]
neur_sample_filter = gaussian_filter(densities_arr[0], (2, 2, 0))[..., :12]

vols = [density_gt_nofilter.flatten(), density_gt_filter.flatten(), ddpm_recon_filter.flatten(), neur_recon_filter.flatten(), wiener_recon_filter.flatten()]

In [ ]:
for vol in vols: 
    print(vol.max(), vol.min())

In [ ]:
# ---- make a 1x4 3D subplot canvas ----
fig = make_subplots(
    rows=1, cols=5,
    specs=[[{"type": "scene"}]*5],
    horizontal_spacing=0.01,
    # subplot_titles=("Ground Truth", "Ground Truth (Blurred)", "Ours", "Neural Ensemble", "Wiener Filter")  # edit or remove as you like
    subplot_titles=("", "", "", "", "")
)

# ---- add four identical Volume traces (you'll swap data per panel later) ----
for c in range(1, 6):
    if c == 1: 
        isomin = 1.5
    else:
        isomin = 0.5
    fig.add_trace(
        go.Volume(
            x=Z.flatten(), y=Y.flatten(), z=X.flatten(),
            value=vols[c-1],
            isomin=isomin,
            opacity=0.15,
            surface_count=12,
            showscale=False,            # keep off (set True on one if you want a colorbar)
            colorscale='Inferno'
        ),
        row=1, col=c
    )

# ---- shared scene styling for each subplot ----
scene_layout = dict(
    aspectmode='manual',
    aspectratio=dict(x=1, y=1, z=1),
    xaxis=dict(showticklabels=False, title_text='Redshift'),
    yaxis=dict(showticklabels=False, title_text='RA'),
    zaxis=dict(showticklabels=False, title_text='DEC')
)

# Apply to scene, scene2, scene3, scene4
fig.update_layout(
    scene=scene_layout,
    scene2=scene_layout,
    scene3=scene_layout,
    scene4=scene_layout,
    scene5=scene_layout,
    margin=dict(l=0, r=0, t=40, b=0)  # a little top room for subplot titles
)

# ---- helper: zoom out all scenes the same way ----
def zoom_out_all(fig, scale=1.3, default_eye=(1.25, 1.25, 1.25)):
    def _num(v, d):
        try:
            return float(v)
        except (TypeError, ValueError):
            return float(d)

    for i in range(1, 6):
        key = f"scene{i}" if i > 1 else "scene"
        sc = getattr(fig.layout, key, None)
        if sc is None:
            continue

        cam = sc.camera
        e = getattr(cam, "eye", None)

        # Safely coerce each component, fall back if missing/None
        ex = _num(getattr(e, "x", None), default_eye[0])
        ey = _num(getattr(e, "y", None), default_eye[1])
        ez = _num(getattr(e, "z", None), default_eye[2])

        new_eye = dict(x=ex * scale, y=ey * scale, z=ez * scale)
        # Update through layout to avoid object-type mismatches
        fig.update_layout(**{f"{key}_camera": dict(eye=new_eye)})



# Example: 30% wider framing for all four
zoom_out_all(fig, scale=1.3)

# Optional overall figure title (centered)
fig.update_layout(
    title=dict(
        text="",
        x=0.5, xanchor="center",
        y=0.98, yanchor="top",
        font=dict(size=22),
        pad=dict(t=10, b=10)
    )
)

# Make the canvas wider and add more spacing between columns
fig.update_layout(
    width=2400, height=600,                 # wider image for export
    margin=dict(l=20, r=20, t=60, b=20)
)

# 1) Bigger subplot titles (the ones from `subplot_titles=...`)
for ann in fig.layout.annotations:
    ann.update(font=dict(size=28))   # <- pick your size

# 2) Bigger x-axis label ("Redshift") and (optional) x-tick size for all 4 scenes
# def set_xfonts(fig, title_size=24, tick_size=14):
#     for i in range(1, 6):
#         key = f"scene{i}" if i > 1 else "scene"
#         sc = getattr(fig.layout, key, None)
#         if sc is None:
#             continue
#         sc.update(
#             xaxis=dict(title=dict(font=dict(size=title_size))),
#             yaxis=dict(title=dict(font=dict(size=title_size))),
#             zaxis=dict(title=dict(font=dict(size=title_size))),
#         )
def set_xfonts(fig, title_size=24, tick_size=14,
               latex_family="STIX Two Text, Times New Roman, serif"):
    for i in range(1, 6):
        key = f"scene{i}" if i > 1 else "scene"
        sc = getattr(fig.layout, key, None)
        if sc is None:
            continue
        sc.update(
            xaxis=dict(
                title=dict(font=dict(size=title_size, family=latex_family)),
                tickfont=dict(size=tick_size, family=latex_family),
            ),
            yaxis=dict(
                title=dict(font=dict(size=title_size, family=latex_family)),
                tickfont=dict(size=tick_size, family=latex_family),
            ),
            zaxis=dict(
                title=dict(font=dict(size=title_size, family=latex_family)),
                tickfont=dict(size=tick_size, family=latex_family),
            ),
        )

set_xfonts(fig, title_size=28, tick_size=14)

# Save and show
pio.write_image(fig, "../jwst_figs/3d_recon_panel_4.svg", scale=4)
fig.show()


In [ ]:
# ---- make a 1x4 3D subplot canvas ----
fig = make_subplots(
    rows=1, cols=4,
    specs=[[{"type": "scene"}]*4],
    horizontal_spacing=0.01,
    subplot_titles=("Ground Truth (Blurred)", "Ours", "Neural Ensemble", "Wiener Filter")  # edit or remove as you like
)

# ---- add four identical Volume traces (you'll swap data per panel later) ----
for c in range(1, 5):
    isomin = 0.5
    fig.add_trace(
        go.Volume(
            x=Z.flatten(), y=Y.flatten(), z=X.flatten(),
            value=vols[c-1],
            isomin=isomin,
            opacity=0.15,
            surface_count=12,
            showscale=False,            # keep off (set True on one if you want a colorbar)
            colorscale='Inferno'
        ),
        row=1, col=c
    )

# ---- shared scene styling for each subplot ----
scene_layout = dict(
    aspectmode='manual',
    aspectratio=dict(x=1.5, y=1, z=1),
    xaxis=dict(showticklabels=False, title_text='Redshift'),
    yaxis=dict(showticklabels=False, title_text='RA'),
    zaxis=dict(showticklabels=False, title_text='DEC')
)

# Apply to scene, scene2, scene3, scene4
fig.update_layout(
    scene=scene_layout,
    scene2=scene_layout,
    scene3=scene_layout,
    scene4=scene_layout,
    margin=dict(l=0, r=0, t=40, b=0)  # a little top room for subplot titles
)

# ---- helper: zoom out all scenes the same way ----
def zoom_out_all(fig, scale=1.3, default_eye=(1.25, 1.25, 1.25)):
    def _num(v, d):
        try:
            return float(v)
        except (TypeError, ValueError):
            return float(d)

    for i in range(1, 5):
        key = f"scene{i}" if i > 1 else "scene"
        sc = getattr(fig.layout, key, None)
        if sc is None:
            continue

        cam = sc.camera
        e = getattr(cam, "eye", None)

        # Safely coerce each component, fall back if missing/None
        ex = _num(getattr(e, "x", None), default_eye[0])
        ey = _num(getattr(e, "y", None), default_eye[1])
        ez = _num(getattr(e, "z", None), default_eye[2])

        new_eye = dict(x=ex * scale, y=ey * scale, z=ez * scale)
        # Update through layout to avoid object-type mismatches
        fig.update_layout(**{f"{key}_camera": dict(eye=new_eye)})



# Example: 30% wider framing for all four
zoom_out_all(fig, scale=1.3)

# Optional overall figure title (centered)
fig.update_layout(
    title=dict(
        text="",
        x=0.5, xanchor="center",
        y=0.98, yanchor="top",
        font=dict(size=22),
        pad=dict(t=10, b=10)
    )
)

# Make the canvas wider and add more spacing between columns
fig.update_layout(
    width=2200, height=600,                 # wider image for export
    margin=dict(l=20, r=20, t=60, b=20)
)

# 1) Bigger subplot titles (the ones from `subplot_titles=...`)
for ann in fig.layout.annotations:
    ann.update(font=dict(size=28))   # <- pick your size

# 2) Bigger x-axis label ("Redshift") and (optional) x-tick size for all 4 scenes
def set_xfonts(fig, title_size=24, tick_size=14):
    for i in range(1, 5):
        key = f"scene{i}" if i > 1 else "scene"
        sc = getattr(fig.layout, key, None)
        if sc is None:
            continue
        sc.update(
            xaxis=dict(title=dict(font=dict(size=title_size))),
            yaxis=dict(title=dict(font=dict(size=title_size))),
            zaxis=dict(title=dict(font=dict(size=title_size))),
        )

set_xfonts(fig, title_size=20, tick_size=14)

# Save and show
pio.write_image(fig, "../jwst_figs/gt_panel.png")
fig.show()


In [ ]:
fig = go.Figure(data=go.Volume(
    x=Z.flatten(),y=Y.flatten(), z=X.flatten(),
    value=density_gt_filter.flatten(),
    isomin=1.0,
    opacity=0.15,
    surface_count=12,
    showscale=False
    ))

fig.update_layout(scene_aspectmode='manual', 
                  scene_aspectratio=dict(x=1.5, y=1, z=1))

fig.update_scenes(
    xaxis_showticklabels=False, yaxis_showticklabels=False, zaxis_showticklabels=False,
    xaxis_title_text='Redshift', yaxis_title_text='', zaxis_title_text=''
)

fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=0),   # kill outer margins
    # paper_bgcolor='rgba(0,0,0,0)'      # transparent background (helps if you post-trim)
)


# Example: 30% wider framing
zoom_out_same_pov(fig, scale=1.3)

fig.update_traces(colorscale='Inferno') 

fig.update_layout(
    title=dict(
        text="Ground Truth",
        x=0.5,                 # center
        xanchor="center",
        y=0.95,                # a bit lower than the very top (optional)
        yanchor="top",
        font=dict(size=22),
        pad=dict(t=10, b=10)
    )
)
pio.write_image(fig, "../jwst_figs/gt.png")

fig.show()

In [ ]:
fig = go.Figure(data=go.Volume(
    x=Z.flatten(),y=Y.flatten(), z=X.flatten(),
    value=wiener_recon_filter.flatten(),
    isomin=0.5,
    opacity=0.15,
    surface_count=12,
    showscale=False
    ))

fig.update_layout(scene_aspectmode='manual', 
                  scene_aspectratio=dict(x=1.5, y=1, z=1))

fig.update_scenes(
    xaxis_showticklabels=False, yaxis_showticklabels=False, zaxis_showticklabels=False,
    xaxis_title_text='Redshift', yaxis_title_text='', zaxis_title_text=''
)

fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=0),   # kill outer margins
    # paper_bgcolor='rgba(0,0,0,0)'      # transparent background (helps if you post-trim)
)


# Example: 30% wider framing
zoom_out_same_pov(fig, scale=1.3)

fig.update_traces(colorscale='Inferno') 

fig.update_layout(
    title=dict(
        text="Wiener Filter",
        x=0.5,                 # center
        xanchor="center",
        y=0.95,                # a bit lower than the very top (optional)
        yanchor="top",
        font=dict(size=22),
        pad=dict(t=10, b=10)
    )
)
pio.write_image(fig, "../jwst_figs/wiener.png")
fig.show()

In [ ]:
fig = go.Figure(data=go.Volume(
    x=Z.flatten(),y=Y.flatten(), z=X.flatten(),
    value=ddpm_sample_filter.flatten(),
    isomin=0.5,
    opacity=0.15,
    surface_count=12,
    showscale=False
    ))

fig.update_layout(scene_aspectmode='manual', 
                  scene_aspectratio=dict(x=1.5, y=1, z=1))


fig.update_scenes(
    xaxis_showticklabels=False, yaxis_showticklabels=False, zaxis_showticklabels=False,
    xaxis_title_text='Redshift', yaxis_title_text='', zaxis_title_text=''
)

fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=0),   # kill outer margins
    # paper_bgcolor='rgba(0,0,0,0)'      # transparent background (helps if you post-trim)
)


# Example: 30% wider framing
zoom_out_same_pov(fig, scale=1.3)

fig.update_traces(colorscale='Inferno') 

fig.update_layout(
    title=dict(
        text="Ours",
        x=0.5,                 # center
        xanchor="center",
        y=0.95,                # a bit lower than the very top (optional)
        yanchor="top",
        font=dict(size=22),
        pad=dict(t=10, b=10)
    )
)
# pio.write_image(fig, "../jwst_figs/ddpm.png")

fig.show()

In [ ]:
fig = go.Figure(data=go.Volume(
    x=Z.flatten(),y=Y.flatten(), z=X.flatten(),
    value=neur_sample_filter.flatten(),
    isomin=0.5,
    opacity=0.15,
    surface_count=12,
    showscale=False
    ))

fig.update_layout(scene_aspectmode='manual', 
                  scene_aspectratio=dict(x=1.5, y=1, z=1))


fig.update_scenes(
    xaxis_showticklabels=False, yaxis_showticklabels=False, zaxis_showticklabels=False,
    xaxis_title_text='Redshift', yaxis_title_text='', zaxis_title_text=''
)

fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=0),   # kill outer margins
    # paper_bgcolor='rgba(0,0,0,0)'      # transparent background (helps if you post-trim)
)


# Example: 30% wider framing
zoom_out_same_pov(fig, scale=1.3)

fig.update_traces(colorscale='Inferno') 

fig.update_layout(
    title=dict(
        text="Neural Recon",
        x=0.5,                 # center
        xanchor="center",
        y=0.95,                # a bit lower than the very top (optional)
        yanchor="top",
        font=dict(size=22),
        pad=dict(t=10, b=10)
    )
)
# pio.write_image(fig, "../jwst_figs/neur.png")

fig.show()